In [0]:
import requests
import json

print("Conectando na Torre de Controle (OpenSky API)...")

# Coordenadas (Bounding Box) focadas na região de Toronto, ON
# lamin (Lat Min), lomin (Lon Min), lamax (Lat Max), lomax (Lon Max)
URL = "https://opensky-network.org/api/states/all?lamin=43.0&lomin=-80.5&lamax=44.5&lomax=-78.5"

resposta = requests.get(URL)

if resposta.status_code == 200:
    dados_voo = resposta.json()

    # A API retorna um dicionário com a chave 'states', que contém a lista de aviões
    avioes = dados_voo.get('states')
    if avioes:
        print("✅ Sucesso! Radar conectado.")
        print((f"Número de aeronaves detectadas no radar de Toronto agora: {len(avioes)}"))

    # Mostrando como a API mostra os dados do primeiro avião da lista
        print("\nEstrutura bruta do primeiro avião detectado:")
        print(json.dumps(avioes[0], indent=4))

    else:
        print("✅ Radar conectado, mas o céu está vazio nesta pequena região agora. Tente novamente em alguns minutos!")
else:
    print(f"❌ Erro ao acessar a API! Código de Status: {resposta.status_code}")


Conectando na Torre de Controle (OpenSky API)...
✅ Sucesso! Radar conectado.
Número de aeronaves detectadas no radar de Toronto agora: 60

Estrutura bruta do primeiro avião detectado:
[
    "c02e9b",
    "JZA48   ",
    "Canada",
    1772543334,
    1772543346,
    -79.6176,
    43.6827,
    60.96,
    true,
    0,
    174.38,
    null,
    null,
    null,
    "2241",
    false,
    0
]


# CAMADA BRONZE/SILVER

In [0]:
import requests
import json
import pandas as pd

print("Buscando dados no radar de Toronto")

URL = "https://opensky-network.org/api/states/all?lamin=43.0&lomin=-80.5&lamax=44.5&lomax=-78.5"
resposta = requests.get(URL) 

if resposta.status_code == 200:
    dados_voo = resposta.json()
    avioes = dados_voo.get('states')

    if avioes:
        print(f"✅ {len(avioes)} aviões detectados! Estruturando no PySpark...")

        # O "Gabarito" da OpenSky: Os nomes exatos de cada item da lista que eles mandam
        colunas_radar = [
            "icao24", "callsign", "origin_country", "time_position", 
            "last_contact", "longitude", "latitude", "baro_altitude", 
            "on_ground", "velocity", "true_track", "vertical_rate", 
            "sensors", "geo_altitude", "squawk", "spi", "position_source"
        ]


        df_pandas = pd.DataFrame(avioes, columns=colunas_radar)
        # Mágica do PySpark: Transformamos a lista de listas em uma tabela com as colunas nomeadas
        df_radar = spark.createDataFrame(df_pandas)
        # Limpeza Rápida (Silver): Filtrando apenas aviões que NÃO estão no chão e removendo nulos
        df_voando = df_radar.filter("on_ground == false AND latitude IS NOT NULL")

        # Exibindo nossa tabela de radar estruturada!
        display(df_voando)

    else:
        print("Céu vazio na região neste exato segundo")
else:
    print(f"Erro na API: {resposta.status_code}")

Buscando dados no radar de Toronto
✅ 60 aviões detectados! Estruturando no PySpark...


icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
c02e91,JZA781,Canada,1772543377,1772543378,-79.2185,43.222,4549.14,false,164.95,309.05,-9.75,null,4556.76,7027,false,0
c0716c,CGQYT,Canada,1772543378,1772543378,-80.2704,43.294,1249.68,false,54.54,334.89,1.3,null,1257.3,1200,false,0
a86925,EDV5140,United States,1772543378,1772543378,-79.8066,43.5766,1021.08,false,119.32,44.65,-3.25,null,1043.94,4157,false,0
adc95d,FDX156,United States,1772543378,1772543378,-78.7884,43.7695,11277.6,false,255.78,49.49,0.0,null,11193.78,1614,false,0
c06229,CGLEM,Canada,1772543378,1772543378,-80.1565,44.2753,8534.4,false,150.73,242.12,0.33,null,8481.06,0677,false,0
c0635f,PTR2461,Canada,1772543378,1772543378,-79.1554,43.7145,3147.06,false,105.93,60.95,16.26,null,3131.82,2222,false,0
c049fd,WJA382,Canada,1772543377,1772543377,-79.6601,43.6358,289.56,false,71.33,46.75,-3.9,null,358.14,7103,false,0
adeddd,AAL1328,United States,1772543378,1772543378,-79.8692,43.0226,10972.8,false,183.67,269.36,0.0,null,10919.46,7311,false,0
c0405a,ROU1931,Canada,1772543378,1772543378,-79.7543,43.8502,3246.12,false,139.79,286.9,10.73,null,3238.5,0546,false,0
c05ef7,JZA7953,Canada,1772543377,1772543377,-79.472,43.6062,373.38,false,69.76,28.64,-4.23,null,441.96,6233,false,0


In [0]:
# Salvando nossa tabela como a visão atual do radar
df_voando.write.mode("overwrite").saveAsTable("gold_radar_toronto")
print("Tabela gold_radar_toronto salva e pronta para a Torre de Controle")

Tabela gold_radar_toronto salva e pronta para a Torre de Controle
